In [1]:
import os
import time
import numpy as np
import pandas as pd
from pathlib import Path
from multiprocessing import Pool, cpu_count
from functools import partial

# Internal framework imports
from src.config import ROOT_DIR, PROCESSED_IMG_DIR
from src.utils import get_all_image_paths, seprate_processed_files, get_skeleton_path
from src.feature_extraction import (
    _extract_new_features_for_image_rois,
    build_new_feature_column_names
)

# Setup I/O Directories
CSV_DIR = Path(os.path.join(ROOT_DIR, "notebooks/csv_files/subset_1498images"))
COORDS_CSV_PATH = CSV_DIR / "roi_coordinates_subset_b.csv"
OLD_FEATURES_CSV_PATH = CSV_DIR / "roi_features_subset_b.csv"
FINAL_MERGED_CSV_PATH = CSV_DIR / "roi_all_features_combined.csv"

print(f"Target CSV Location: {CSV_DIR}")

Target CSV Location: C:\Users\ferra\MIC\1r_any_UNICAS\2n_Semestre\Image_Processing_and_Analysis\project\MIC_project\Proposal_StenosisDetection\notebooks\csv_files\subset_1498images


In [ ]:
# 1. Load the ground-truth coordinate reference sheet
try:
    df_coords = pd.read_csv(COORDS_CSV_PATH)
    print(f"Loaded {len(df_coords)} ROI coordinate paths spanning {df_coords['image_name'].nunique()} distinct images.")
except Exception:
    print(f"ROI coordinates file in {COORDS_CSV_PATH} was not found. Check that it exists in such directory.")

# 2. Match unique images against underlying directory paths to prevent filesystem overhead in workers
all_paths = get_all_image_paths(directory=PROCESSED_IMG_DIR)
separated = seprate_processed_files(all_paths)
processed_paths = separated[0]

# Structure mapping dictionary: image_name string -> file paths
path_lookup = {}
for path in processed_paths:
    p = Path(path)
    patient_id = p.parts[-3]
    serie_id   = p.parts[-2]
    frame_stem = p.stem.replace('_processed', '')[-4:]
    image_name = f"{patient_id}_{serie_id}_{frame_stem}"

    mask_path = get_skeleton_path(path).replace('_skeleton.png', '_mask.png')
    path_lookup[image_name] = {
        'img': path,
        'mask': mask_path
    }

# 3. Batch coordinates by image frame for optimized worker distributions
image_groups = []
for img_name, group_df in df_coords.groupby('image_name', sort=False):
    if img_name in path_lookup:
        image_groups.append((img_name, group_df, path_lookup[img_name]))

print(f"Prepared batch execution inputs for {len(image_groups)} images matching coordinate references.")

In [ ]:
# Setup process configuration
n_cores = cpu_count()
worker_fn = partial(_extract_new_features_for_image_rois, roi_size=80)

print(f"Launching feature calculation pipeline over {n_cores} parallel logical processors...")
start_time = time.time()

with Pool(processes=n_cores) as pool:
    results = pool.map(worker_fn, image_groups)

# Collapse nested worker iterations to a continuous linear list
extracted_rows = [row for img_results in results for row in img_results]
elapsed = time.time() - start_time

df_new_features = pd.DataFrame(extracted_rows)
print(f"\nCompleted extraction matrix in {elapsed:.2f} seconds.")
print(f"Extracted shape: {df_new_features.shape}")

In [ ]:
# Load your original full matrix containing old Gabor and tile stats
try:
    df_old = pd.read_csv(OLD_FEATURES_CSV_PATH)
    print(f"Base Features Shape: {df_old.shape}")
except Exception:
    print(f"ROI old features file in {OLD_FEATURES_CSV_PATH} was not found. Check that it exists in such directory.")

# Drop target label column from the new side to prevent duplicated columns during merge
df_new_subset = df_new_features.drop(columns=['image_name', 'label'])

# Perform precise inner-join intersection matching exactly on the primary key: 'roi_name'
df_combined = pd.merge(df_old, df_new_subset, on='roi_name', how='inner')

print(f"\nMerged Output Matrix verification dimensions: {df_combined.shape}")
print(f"Stenosis Targets Count: {df_combined['label'].sum()}")
print(f"Healthy Targets Count: {(df_combined['label'] == 0).sum()}")

# Export final data pipeline asset
df_combined.to_csv(FINAL_MERGED_CSV_PATH, index=False)
print(f"\nSuccess! Operational Dataset exported and ready for training at:\n -> {FINAL_MERGED_CSV_PATH}")